# 🧬 TechCorp AI — Fine-tuning LoRA médical (Google Colab)

Modèle **médical expérimental** (mission R&D — pas pour la production).

**Avant de lancer** : `Exécution > Modifier le type d'exécution > GPU (T4)`.

Étapes : install → GPU → dataset → modèle+LoRA → entraînement → test → export.

## 1. Installation des dépendances

In [ ]:
!pip -q install "transformers>=4.40" "peft>=0.10" "datasets>=2.18" "accelerate>=0.27" bitsandbytes trl sentencepiece

## 2. Vérification du GPU

In [ ]:
import torch
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('⚠️ Pas de GPU — active GPU dans Exécution > Type d\'exécution.')

## 3. Configuration

In [ ]:
BASE_MODEL   = 'microsoft/phi-2'   # modèle de base léger (~2.7B)
MAX_SAMPLES  = 3000                # nb d'exemples (augmenter si temps OK)
EPOCHS       = 3
MAX_LENGTH   = 512
LORA_R       = 16
LORA_ALPHA   = 32
OUTPUT_DIR   = 'medical_model_lora'

## 4. Chargement du dataset médical

Deux options : (A) télécharger depuis HuggingFace, ou (B) uploader le fichier `medical_dataset_training.json` produit par `prepare_dataset.py` (onglet Fichiers de Colab).

In [ ]:
import os, json
from datasets import load_dataset, Dataset

if os.path.exists('medical_dataset_training.json'):
    print('Option B : dataset local pré-nettoyé')
    data = json.load(open('medical_dataset_training.json', encoding='utf-8'))
    ds = Dataset.from_list([{'text': d['text']} for d in data[:MAX_SAMPLES]])
else:
    print('Option A : téléchargement HuggingFace (streaming)')
    raw = load_dataset('ruslanmv/ai-medical-chatbot', split='train', streaming=True)
    rows = []
    for i, ex in enumerate(raw):
        if i >= MAX_SAMPLES: break
        p, d = (ex.get('Patient') or '').strip(), (ex.get('Doctor') or '').strip()
        if len(p) > 10 and len(d) > 20:
            rows.append({'text': f'<|user|>\n{p}\n<|assistant|>\n{d}<|endoftext|>'})
    ds = Dataset.from_list(rows)

print('Exemples :', len(ds))
print(ds[0]['text'][:200])

## 5. Modèle + quantization 4-bit + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                  lora_dropout=0.05, bias='none',
                  target_modules=['q_proj','k_proj','v_proj','dense'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Tokenisation

In [ ]:
def tok(ex):
    out = tokenizer(ex['text'], truncation=True, max_length=MAX_LENGTH, padding='max_length')
    out['labels'] = out['input_ids'].copy()
    return out

tokenized = ds.map(tok, remove_columns=ds.column_names)
split = tokenized.train_test_split(test_size=0.05, seed=42)
print('Train', len(split['train']), '| Eval', len(split['test']))

## 7. Entraînement

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=True, logging_steps=25,
    eval_strategy='epoch', save_strategy='epoch',
    warmup_steps=100, lr_scheduler_type='cosine', report_to='none')

trainer = Trainer(model=model, args=args,
                  train_dataset=split['train'], eval_dataset=split['test'],
                  data_collator=DataCollatorForSeq2Seq(tokenizer, model, padding=True))
trainer.train()

## 8. Test du modèle fine-tuné

In [ ]:
def ask(q, n=200):
    p = f'<|user|>\n{q}\n<|assistant|>\n'
    ids = tokenizer(p, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=n, temperature=0.7, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.1)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for q in ["J'ai de la fièvre depuis 3 jours, que faire ?",
          'Quels sont les signes d\'un AVC ?']:
    print('Q:', q); print(ask(q)); print('—'*40)

## 9. Sauvegarde & téléchargement des adaptateurs LoRA

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Sauvegardé dans', OUTPUT_DIR)

# Archive téléchargeable (adaptateurs LoRA = quelques Mo)
!zip -rq medical_model_lora.zip {OUTPUT_DIR}
try:
    from google.colab import files
    files.download('medical_model_lora.zip')
except Exception as e:
    print('Téléchargez manuellement medical_model_lora.zip —', e)